# Correlation

In this section we will develop a measure of how tightly clustered a scatter diagram is about a straight line. Formally, this is called measuring *linear association*.

In [ ]:
library(tidyverse)
set.seed(42)

The table `hybrid` contains data on hybrid passenger cars sold in the United States from 1997 to 2013. The columns:

- `vehicle`: model of the car
- `year`: year of manufacture
- `msrp`: manufacturer's suggested retail price in 2013 dollars
- `acceleration`: acceleration rate in km per hour per second
- `mpg`: fuel economy in miles per gallon
- `class`: the model's class.

In [ ]:
hybrid <- read_csv("hybrid.csv")
hybrid

The graph below is a scatter plot of `msrp` *versus* `acceleration`. That means `msrp` is plotted on the vertical axis and `acceleration` on the horizontal.

In [ ]:
ggplot(hybrid, aes(x = acceleration, y = msrp)) +
  geom_point() +
  theme_minimal()

Notice the positive association. The scatter of points is sloping upwards, indicating that cars with greater acceleration tended to cost more, on average.

The scatter diagram of MSRP versus mileage shows a negative association. Hybrid cars with higher mileage tended to cost less, on average.

In [ ]:
ggplot(hybrid, aes(x = mpg, y = msrp)) +
  geom_point() +
  theme_minimal()

Along with the negative association, the scatter diagram of price versus efficiency shows a non-linear relation between the two variables.

If we restrict the data just to the SUV class, however, the association between price and efficiency is still negative but the relation appears to be more linear.

In [ ]:
suv <- hybrid |>
  filter(class == "SUV")

ggplot(suv, aes(x = mpg, y = msrp)) +
  geom_point() +
  theme_minimal()

In [ ]:
ggplot(suv, aes(x = acceleration, y = msrp)) +
  geom_point() +
  theme_minimal()

We could plot all the variables in standard units and the plots would look the same. This gives us a way to compare the degree of linearity in two scatter diagrams.

In [ ]:
suv |>
  mutate(
    mpg_su = (mpg - mean(mpg)) / sd(mpg),
    msrp_su = (msrp - mean(msrp)) / sd(msrp)
  ) |>
  ggplot(aes(x = mpg_su, y = msrp_su)) +
  geom_point() +
  xlim(-3, 3) + ylim(-3, 3) +
  labs(x = "mpg (standard units)", y = "msrp (standard units)") +
  theme_minimal()

In [ ]:
suv |>
  mutate(
    accel_su = (acceleration - mean(acceleration)) / sd(acceleration),
    msrp_su = (msrp - mean(msrp)) / sd(msrp)
  ) |>
  ggplot(aes(x = accel_su, y = msrp_su)) +
  geom_point() +
  xlim(-3, 3) + ylim(-3, 3) +
  labs(x = "acceleration (standard units)", y = "msrp (standard units)") +
  theme_minimal()

## The correlation coefficient

The *correlation coefficient* measures the strength of the linear relationship between two variables. Graphically, it measures how clustered the scatter diagram is around a straight line.

Here are some mathematical facts about $r$:

- The correlation coefficient $r$ is a number between $-1$ and 1.
- $r$ measures the extent to which the scatter plot clusters around a straight line.
- $r = 1$ if the scatter diagram is a perfect straight line sloping upwards, and $r = -1$ if the scatter diagram is a perfect straight line sloping downwards.

The function `r_scatter` takes a value of $r$ as its argument and simulates a scatter plot with a correlation very close to $r$.

In [ ]:
r_scatter <- function(r) {
  n <- 1000
  x <- rnorm(n)
  z <- rnorm(n)
  y <- r * x + sqrt(1 - r^2) * z
  tibble(x = x, y = y) |>
    ggplot(aes(x, y)) +
    geom_point(alpha = 0.5) +
    coord_cartesian(xlim = c(-4, 4), ylim = c(-4, 4)) +
    theme_minimal()
}

In [ ]:
r_scatter(0.9)

In [ ]:
r_scatter(0.25)

In [ ]:
r_scatter(0)

In [ ]:
r_scatter(-0.55)

## Calculating $r$

**Formula for $r$**:

**$r$ is the average of the products of the two variables, when both variables are measured in standard units.**

Here are the steps in the calculation.

In [ ]:
x <- 1:6
y <- c(2, 3, 1, 5, 2, 7)
t <- tibble(x = x, y = y)
t

Based on the scatter diagram, we expect that $r$ will be positive but not equal to 1.

In [ ]:
ggplot(t, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal()

**Step 1.** Convert each variable to standard units.

In [ ]:
t_su <- t |>
  mutate(
    x_su = (x - mean(x)) / sd(x),
    y_su = (y - mean(y)) / sd(y)
  )
t_su

**Step 2.** Multiply each pair of standard units.

In [ ]:
t_product <- t_su |>
  mutate(product_su = x_su * y_su)
t_product

**Step 3.** $r$ is the average of the products computed in Step 2.

In [ ]:
r <- mean(t_product$product_su)
r

As expected, $r$ is positive but not equal to 1.

## Properties of $r$

The calculation shows that:

- $r$ is a pure number. It has no units. This is because $r$ is based on standard units.
- $r$ is unaffected by changing the units on either axis.
- $r$ is unaffected by switching the axes.

In [ ]:
ggplot(t, aes(x = y, y = x)) +
  geom_point(color = "red", size = 3) +
  theme_minimal()

## Using R's built-in `cor()` function

R provides a built-in `cor()` function that computes the Pearson correlation coefficient between two numeric vectors. It works by converting each variable to standard units (subtracting the mean and dividing by the standard deviation), multiplying the standardized values together, and averaging the products — exactly the steps we just performed manually.

One small difference: `cor()` divides by $n - 1$ rather than $n$ when standardizing (using the *sample* standard deviation). This is a standard statistical correction that we will explore in a future assignment where you will program your own correlation function from scratch.

In [ ]:
cor(t$x, t$y)

As we noticed, the order in which the variables are specified doesn't matter.

In [ ]:
cor(t$y, t$x)

Calling `cor()` on the `suv` tibble gives us the correlation between price and mileage as well as the correlation between price and acceleration.

In [ ]:
cor(suv$mpg, suv$msrp)

In [ ]:
cor(suv$acceleration, suv$msrp)

These values confirm what we observed:

- There is a negative association between price and efficiency, whereas the association between price and acceleration is positive.
- The linear relation between price and acceleration is a little weaker (correlation about 0.5) than between price and mileage (correlation about −0.67).

## Association is not Causation

Correlation only measures association. Correlation does not imply causation. Though the correlation between the weight and the math ability of children in a school district may be positive, that does not mean that doing math makes children heavier or that putting on weight improves the children's math skills. Age is a confounding variable.

## Correlation Measures *Linear* Association

Variables that have strong non-linear association might have very low correlation. Here is an example of variables that have a perfect quadratic relation $y = x^2$ but have correlation equal to 0.

In [ ]:
nonlinear <- tibble(
  x = seq(-4, 4, by = 0.5),
  y = x^2
)

ggplot(nonlinear, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal()

In [ ]:
cor(nonlinear$x, nonlinear$y)

## Correlation is Affected by Outliers

Outliers can have a big effect on correlation. Here is an example where $r = 1$ is turned into $r \approx 0$ by adding one outlying point.

In [ ]:
line <- tibble(x = 1:4, y = 1:4)

ggplot(line, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal()

In [ ]:
cor(line$x, line$y)

In [ ]:
outlier <- tibble(x = 1:5, y = c(1, 2, 3, 4, 0))

ggplot(outlier, aes(x, y)) +
  geom_point(color = "red", size = 3) +
  theme_minimal()

In [ ]:
cor(outlier$x, outlier$y)

## Ecological Correlations Should be Interpreted with Care

Correlations based on aggregated data can be misleading.

In [ ]:
sat2014 <- read_csv("sat2014.csv") |>
  arrange(State)
sat2014

In [ ]:
ggplot(sat2014, aes(x = `Critical Reading`, y = Math)) +
  geom_point() +
  theme_minimal()

In [ ]:
cor(sat2014$`Critical Reading`, sat2014$Math)

That's an extremely high correlation. But it's important to note that this does not reflect the strength of the relation between the Math and Critical Reading scores of *students*.

The data consist of average scores in each state. Correlations based on aggregates and averages are called *ecological correlations* and must be interpreted with care.